# test_deployments.ipynb：逐行演算 `envs/grid/deployments.py`

这个 notebook 用小数值例子说明 `deployments.py` 的作用：

- 把电池容量和 C-rate 配置解析成每个 agent 的容量与功率上限。
- 把 agent bus、电池参数和 SoC 合同装配成 `AgentDeployment`。
- 说明它和 `GridCore`、`GridEnv`、安全投影器之间的关系。


## 0. 准备环境和公共解释工具

Notebook 可能不是从项目根目录启动，所以先找到仓库根目录并加入 `sys.path`，避免 `No module named 'envs'`。后面的 `explain(...)` 会把每一步代码、输入、输出和含义放在一张表里。


In [1]:
from pathlib import Path
import sys
from dataclasses import asdict
from types import SimpleNamespace

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "envs").exists() and (candidate / "configs").exists():
            return candidate
    raise RuntimeError("Cannot find project root containing envs/ and configs/.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from envs.grid.deployments import (  # noqa: E402
    DEFAULT_AGENT_BUS_IDS,
    AgentDeployment,
    _resolve_positive_scalar,
    _resolve_positive_vector,
    build_agent_deployments,
    resolve_fixed_battery_spec,
)

pd.set_option("display.max_colwidth", 140)


def explain(rows):
    display(pd.DataFrame(rows, columns=["代码步骤", "本例输入", "中间输出", "解释"]))


def make_cfg(
    *,
    n_agents=3,
    agent_bus_ids=(12, 4, 2),
    battery_capacity=(10.0, 12.0, 8.0),
    max_charge_rate=0.5,
    init_soc=0.55,
    soc_min=0.10,
    soc_max=0.90,
    efficiency=0.93,
):
    env = SimpleNamespace(
        num_agents=n_agents,
        battery_capacity=battery_capacity,
        max_charge_rate=max_charge_rate,
        init_soc=init_soc,
        soc_min=soc_min,
        soc_max=soc_max,
        efficiency=efficiency,
    )
    grid = SimpleNamespace(agent_bus_ids=list(agent_bus_ids))
    return SimpleNamespace(env=env, grid=grid)


explain(
    [
        ["find_project_root(Path.cwd())", str(Path.cwd()), str(PROJECT_ROOT), "定位仓库根目录，保证 notebook 在任意工作目录下都能导入项目代码。"],
        ["sys.path.insert(0, PROJECT_ROOT)", str(PROJECT_ROOT), str(PROJECT_ROOT in map(Path, sys.path)), "把仓库根目录放到 Python 导入搜索路径中。"],
        ["import deployments symbols", "envs.grid.deployments", "AgentDeployment / resolve_fixed_battery_spec / build_agent_deployments", "导入本 notebook 要演算的部署合同函数。"],
        ["make_cfg(...)", "若干标量和列表", "SimpleNamespace(env=..., grid=...)", "构造一个最小 cfg，模拟真实配置对象中 deployments.py 会读取的字段。"],
    ]
)

,代码步骤,本例输入,中间输出,解释
0,find_project_root(Path.cwd()),D:\GithubProject\MADRL_ESS\tests\testnotebook,D:\GithubProject\MADRL_ESS,定位仓库根目录，保证 notebook 在任意工作目录下都能导入项目代码。
1,"sys.path.insert(0, PROJECT_ROOT)",D:\GithubProject\MADRL_ESS,True,把仓库根目录放到 Python 导入搜索路径中。
2,import deployments symbols,envs.grid.deployments,AgentDeployment / resolve_fixed_battery_spec / build_agent_deployments,导入本 notebook 要演算的部署合同函数。
3,make_cfg(...),若干标量和列表,"SimpleNamespace(env=..., grid=...)",构造一个最小 cfg，模拟真实配置对象中 deployments.py 会读取的字段。


## 1. `AgentDeployment`：单个 agent 的电网部署合同

`AgentDeployment` 本身不做计算，它只是描述一个 agent 接在哪个 bus，以及它对应的固定电池参数。


In [2]:
deployment = AgentDeployment(
    bus_id=12,
    battery_capacity_kwh=10.0,
    battery_power_kw=5.0,
    init_soc=0.55,
    soc_min=0.10,
    soc_max=0.90,
    efficiency=0.93,
)

explain(
    [
        ["AgentDeployment(...)", "bus_id=12, capacity=10 kWh, power=5 kW", asdict(deployment), "生成一个不可变 dataclass，作为后续 GridCore 的 agent 部署输入。"],
        ["deployment.bus_id", "deployment", deployment.bus_id, "GridCore 主要用它决定这个 agent 写入 pandapower 网络的哪个 bus。"],
        ["deployment.battery_power_kw", "10 kWh * 0.5 C", deployment.battery_power_kw, "功率上限来自容量乘以 C-rate，单位是 kW。"],
    ]
)

,代码步骤,本例输入,中间输出,解释
0,AgentDeployment(...),"bus_id=12, capacity=10 kWh, power=5 kW","{'bus_id': 12, 'battery_capacity_kwh': 10.0, 'battery_power_kw': 5.0, 'init_soc': 0.55, 'soc_min': 0.1, 'soc_max': 0.9, 'efficiency': 0.93}",生成一个不可变 dataclass，作为后续 GridCore 的 agent 部署输入。
1,deployment.bus_id,deployment,12,GridCore 主要用它决定这个 agent 写入 pandapower 网络的哪个 bus。
2,deployment.battery_power_kw,10 kWh * 0.5 C,5.0,功率上限来自容量乘以 C-rate，单位是 kW。


## 2. `_resolve_positive_scalar(value, name)`：解析正标量

这个私有函数用于保证像 `max_charge_rate` 这样的配置是正数。它是一个边界检查：坏配置应该在进入环境前失败。


In [3]:
valid_scalar = _resolve_positive_scalar("0.5", name="max_charge_rate")
try:
    _resolve_positive_scalar(0.0, name="max_charge_rate")
except ValueError as exc:
    zero_error = str(exc)
try:
    _resolve_positive_scalar("bad", name="max_charge_rate")
except ValueError as exc:
    text_error = str(exc)

explain(
    [
        ["float(value)", 'value="0.5"', valid_scalar, "字符串数值会被转成 float，后续统一按数值计算。"],
        ["if scalar <= 0.0", "value=0.0", zero_error, "0 或负数没有物理意义，所以直接报错。"],
        ["except (TypeError, ValueError)", 'value="bad"', text_error, "不能转成 float 的配置也会报错，并指出字段名。"],
    ]
)

,代码步骤,本例输入,中间输出,解释
0,float(value),"value=""0.5""",0.5,字符串数值会被转成 float，后续统一按数值计算。
1,if scalar <= 0.0,value=0.0,"max_charge_rate must be positive, got 0.0.",0 或负数没有物理意义，所以直接报错。
2,"except (TypeError, ValueError)","value=""bad""","max_charge_rate must be a positive scalar, got 'bad'.",不能转成 float 的配置也会报错，并指出字段名。


## 3. `_resolve_positive_vector(value, n_agents, name)`：把容量配置变成逐 agent 列表

数学例子：系统有 3 个 agent。容量可以写成一个标量，也可以写成每个 agent 一个值。


In [4]:
scalar_capacity = _resolve_positive_vector(10.0, n_agents=3, name="battery_capacity")
single_item_capacity = _resolve_positive_vector([8.0], n_agents=3, name="battery_capacity")
per_agent_capacity = _resolve_positive_vector([10.0, 12.0, 8.0], n_agents=3, name="battery_capacity")
legacy_equal_extra = _resolve_positive_vector([7.0, 7.0, 7.0, 7.0], n_agents=3, name="battery_capacity")
try:
    _resolve_positive_vector([10.0, 12.0], n_agents=3, name="battery_capacity")
except ValueError as exc:
    mismatch_error = str(exc)

explain(
    [
        ["else: values = [scalar]", "10.0, n_agents=3", scalar_capacity, "标量表示所有 agent 使用同一容量，所以广播成 3 个 10 kWh。"],
        ["if len(values) == 1", "[8.0], n_agents=3", single_item_capacity, "长度为 1 的列表也按标量处理，广播成每个 agent 8 kWh。"],
        ["elif len(values) == n_agents", "[10.0, 12.0, 8.0]", per_agent_capacity, "长度等于 agent 数量时，逐 agent 使用对应容量。"],
        ["当前兼容行为", "[7.0, 7.0, 7.0, 7.0], n_agents=3", legacy_equal_extra, "当前代码会把多余但完全相同的值压成 3 个；这是可优化点，不是推荐新合同。"],
        ["raise ValueError", "[10.0, 12.0], n_agents=3", mismatch_error, "长度不等且不是同一个值时，会报错，避免 agent 数量和容量配置错位。"],
    ]
)

,代码步骤,本例输入,中间输出,解释
0,else: values = [scalar],"10.0, n_agents=3","[10.0, 10.0, 10.0]",标量表示所有 agent 使用同一容量，所以广播成 3 个 10 kWh。
1,if len(values) == 1,"[8.0], n_agents=3","[8.0, 8.0, 8.0]",长度为 1 的列表也按标量处理，广播成每个 agent 8 kWh。
2,elif len(values) == n_agents,"[10.0, 12.0, 8.0]","[10.0, 12.0, 8.0]",长度等于 agent 数量时，逐 agent 使用对应容量。
3,当前兼容行为,"[7.0, 7.0, 7.0, 7.0], n_agents=3","[7.0, 7.0, 7.0]",当前代码会把多余但完全相同的值压成 3 个；这是可优化点，不是推荐新合同。
4,raise ValueError,"[10.0, 12.0], n_agents=3","battery_capacity should provide 3 value(s) for fixed mode, got 2.",长度不等且不是同一个值时，会报错，避免 agent 数量和容量配置错位。


## 4. `resolve_fixed_battery_spec(...)`：容量乘以 C-rate 得到功率上限

这是 `deployments.py` 的核心数学计算。固定电池模式下：

`p_max_kw[i] = capacity_kwh[i] * max_charge_rate`

比如容量 `[10, 12, 8]` kWh，C-rate 为 `0.5`，则功率上限是 `[5, 6, 4]` kW。


In [5]:
capacity_kwh, c_rate, p_max_kw = resolve_fixed_battery_spec(
    [10.0, 12.0, 8.0],
    0.5,
    n_agents=3,
)
manual_p_max_kw = [capacity * c_rate for capacity in capacity_kwh]
per_agent_table = pd.DataFrame(
    {
        "agent_id": [0, 1, 2],
        "capacity_kwh": capacity_kwh,
        "c_rate": [c_rate] * 3,
        "manual_capacity_times_c_rate": manual_p_max_kw,
        "function_p_max_kw": p_max_kw,
    }
)

explain(
    [
        ["capacity_kwh = _resolve_positive_vector(...)", "[10, 12, 8], n_agents=3", capacity_kwh, "先把容量配置整理成逐 agent 的 kWh 列表。"],
        ["c_rate = _resolve_positive_scalar(...)", "0.5", c_rate, "再确认 max_charge_rate 是一个正标量 C-rate。"],
        ["p_max_kw = capacity * c_rate", "[10, 12, 8] * 0.5", p_max_kw, "逐 agent 计算功率上限：10*0.5=5，12*0.5=6，8*0.5=4。"],
    ]
)
display(per_agent_table)

,代码步骤,本例输入,中间输出,解释
0,capacity_kwh = _resolve_positive_vector(...),"[10, 12, 8], n_agents=3","[10.0, 12.0, 8.0]",先把容量配置整理成逐 agent 的 kWh 列表。
1,c_rate = _resolve_positive_scalar(...),0.5,0.5,再确认 max_charge_rate 是一个正标量 C-rate。
2,p_max_kw = capacity * c_rate,"[10, 12, 8] * 0.5","[5.0, 6.0, 4.0]",逐 agent 计算功率上限：10*0.5=5，12*0.5=6，8*0.5=4。


,agent_id,capacity_kwh,c_rate,manual_capacity_times_c_rate,function_p_max_kw
0,0,10.0,0.5,5.0,5.0
1,1,12.0,0.5,6.0,6.0
2,2,8.0,0.5,4.0,4.0


## 5. `build_agent_deployments(cfg)`：从配置装配成 GridCore 输入

这个函数把 `cfg.env` 和 `cfg.grid` 里的字段合并，生成 `GridCore` 可以直接使用的 `AgentDeployment` 列表。


In [6]:
cfg = make_cfg(
    n_agents=3,
    agent_bus_ids=[12, 4, 2],
    battery_capacity=[10.0, 12.0, 8.0],
    max_charge_rate=0.5,
    init_soc=0.55,
    soc_min=0.10,
    soc_max=0.90,
    efficiency=0.93,
)
deployments = build_agent_deployments(cfg)
deployment_table = pd.DataFrame(asdict(item) for item in deployments)

explain(
    [
        ["n_agents = int(cfg.env.num_agents)", "cfg.env.num_agents = 3", 3, "决定要生成几个 AgentDeployment。"],
        ["bus_ids = list(cfg.grid.agent_bus_ids)", "[12, 4, 2]", [12, 4, 2], "每个 agent 对应一个电网 bus。"],
        ["resolve_fixed_battery_spec(...)", "capacity=[10,12,8], C-rate=0.5", "capacity=[10,12,8], power=[5,6,4]", "把电池配置转换为容量和功率上限。"],
        ["for idx in range(n_agents)", "idx=0,1,2", "3 个 AgentDeployment", "逐 agent 拼装 bus、容量、功率、SoC 边界和效率。"],
    ]
)
display(deployment_table)

,代码步骤,本例输入,中间输出,解释
0,n_agents = int(cfg.env.num_agents),cfg.env.num_agents = 3,3,决定要生成几个 AgentDeployment。
1,bus_ids = list(cfg.grid.agent_bus_ids),"[12, 4, 2]","[12, 4, 2]",每个 agent 对应一个电网 bus。
2,resolve_fixed_battery_spec(...),"capacity=[10,12,8], C-rate=0.5","capacity=[10,12,8], power=[5,6,4]",把电池配置转换为容量和功率上限。
3,for idx in range(n_agents),"idx=0,1,2",3 个 AgentDeployment,逐 agent 拼装 bus、容量、功率、SoC 边界和效率。


,bus_id,battery_capacity_kwh,battery_power_kw,init_soc,soc_min,soc_max,efficiency
0,12,10.0,5.0,0.55,0.1,0.9,0.93
1,4,12.0,6.0,0.55,0.1,0.9,0.93
2,2,8.0,4.0,0.55,0.1,0.9,0.93


## 6. `DEFAULT_AGENT_BUS_IDS`：没有显式 bus 配置时的默认接入点

如果 `cfg.grid.agent_bus_ids` 是空列表，`build_agent_deployments` 会使用 `DEFAULT_AGENT_BUS_IDS` 的前 `n_agents` 个 bus。


In [7]:
default_cfg = make_cfg(
    n_agents=3,
    agent_bus_ids=[],
    battery_capacity=5.0,
    max_charge_rate=0.2,
)
default_deployments = build_agent_deployments(default_cfg)
default_table = pd.DataFrame(asdict(item) for item in default_deployments)

explain(
    [
        ["list(cfg.grid.agent_bus_ids)", "[]", [], "显式 bus 配置为空。"],
        ["or list(DEFAULT_AGENT_BUS_IDS[:n_agents])", f"DEFAULT_AGENT_BUS_IDS={DEFAULT_AGENT_BUS_IDS}, n_agents=3", list(DEFAULT_AGENT_BUS_IDS[:3]), "使用默认列表的前三个 bus。"],
        ["battery_capacity=5.0, C-rate=0.2", "5.0 * 0.2", [1.0, 1.0, 1.0], "标量容量和 C-rate 会广播给三个 agent，功率上限都是 1 kW。"],
    ]
)
display(default_table)

,代码步骤,本例输入,中间输出,解释
0,list(cfg.grid.agent_bus_ids),[],[],显式 bus 配置为空。
1,or list(DEFAULT_AGENT_BUS_IDS[:n_agents]),"DEFAULT_AGENT_BUS_IDS=(10, 6, 12, 4, 2), n_agents=3","[10, 6, 12]",使用默认列表的前三个 bus。
2,"battery_capacity=5.0, C-rate=0.2",5.0 * 0.2,"[1.0, 1.0, 1.0]",标量容量和 C-rate 会广播给三个 agent，功率上限都是 1 kW。


,bus_id,battery_capacity_kwh,battery_power_kw,init_soc,soc_min,soc_max,efficiency
0,10,5.0,1.0,0.55,0.1,0.9,0.93
1,6,5.0,1.0,0.55,0.1,0.9,0.93
2,12,5.0,1.0,0.55,0.1,0.9,0.93


## 7. 错误输入：deployments.py 在哪些边界会失败

这些例子展示配置错误如何被拦截。最后一个 `agent_bus_ids` 太短目前会变成 `IndexError`，这也是值得优化成更清楚 `ValueError` 的地方。


In [8]:
def capture_error(label, fn):
    try:
        fn()
    except Exception as exc:
        return {
            "配置场景": label,
            "错误类型": type(exc).__name__,
            "错误消息": str(exc),
        }
    return {"配置场景": label, "错误类型": "无", "错误消息": "没有报错"}


error_rows = [
    capture_error(
        "battery_capacity = -1.0",
        lambda: resolve_fixed_battery_spec(-1.0, 0.5, n_agents=2),
    ),
    capture_error(
        "battery_capacity = [10, 12], n_agents = 3",
        lambda: resolve_fixed_battery_spec([10.0, 12.0], 0.5, n_agents=3),
    ),
    capture_error(
        "max_charge_rate = [0.5, 0.5]",
        lambda: resolve_fixed_battery_spec([10.0, 12.0], [0.5, 0.5], n_agents=2),
    ),
    capture_error(
        "agent_bus_ids = [12], n_agents = 2",
        lambda: build_agent_deployments(
            make_cfg(n_agents=2, agent_bus_ids=[12], battery_capacity=[10.0, 12.0], max_charge_rate=0.5)
        ),
    ),
]

explain(
    [
        ["resolve_fixed_battery_spec(-1.0, ...)", "容量为负", "ValueError", "容量必须是正数。"],
        ["resolve_fixed_battery_spec([10,12], ..., n_agents=3)", "容量数量不足", "ValueError", "逐 agent 容量数量必须和 agent 数量匹配，除非是标量或长度 1。"],
        ["resolve_fixed_battery_spec(..., max_charge_rate=[0.5,0.5])", "C-rate 是列表", "ValueError", "固定电池模式要求 max_charge_rate 是一个统一正标量。"],
        ["build_agent_deployments(cfg)", "bus 数量少于 agent 数量", "IndexError（当前行为）", "这里建议后续优化成显式 ValueError，让错误信息更清楚。"],
    ]
)
display(pd.DataFrame(error_rows))

,代码步骤,本例输入,中间输出,解释
0,"resolve_fixed_battery_spec(-1.0, ...)",容量为负,ValueError,容量必须是正数。
1,"resolve_fixed_battery_spec([10,12], ..., n_agents=3)",容量数量不足,ValueError,逐 agent 容量数量必须和 agent 数量匹配，除非是标量或长度 1。
2,"resolve_fixed_battery_spec(..., max_charge_rate=[0.5,0.5])",C-rate 是列表,ValueError,固定电池模式要求 max_charge_rate 是一个统一正标量。
3,build_agent_deployments(cfg),bus 数量少于 agent 数量,IndexError（当前行为）,这里建议后续优化成显式 ValueError，让错误信息更清楚。


,配置场景,错误类型,错误消息
0,battery_capacity = -1.0,ValueError,"battery_capacity must be positive, got -1.0."
1,"battery_capacity = [10, 12], n_agents = 3",ValueError,"battery_capacity should provide 3 value(s) for fixed mode, got 2."
2,"max_charge_rate = [0.5, 0.5]",ValueError,"fixed battery mode expects max_charge_rate to be a positive scalar C-rate, got [0.5, 0.5]."
3,"agent_bus_ids = [12], n_agents = 2",IndexError,list index out of range


## 8. 和其他模块的关系

通俗地说，`deployments.py` 是“把配置单翻译成电网部署清单”的地方。它不读数据、不跑潮流、不训练 DRL；它只把 `cfg` 里的 agent bus 和固定电池参数整理成明确合同。


In [9]:
relationship = pd.DataFrame(
    [
        {
            "模块": "cfg.env / cfg.grid",
            "输入或输出": "输入",
            "在本例中的值": "num_agents=3, bus=[12,4,2], capacity=[10,12,8], C-rate=0.5",
            "作用": "提供配置原料。",
        },
        {
            "模块": "envs.grid.deployments",
            "输入或输出": "转换层",
            "在本例中的值": "3 个 AgentDeployment",
            "作用": "把配置整理成每个 agent 的电网接入点和电池合同。",
        },
        {
            "模块": "GridCore",
            "输入或输出": "消费者",
            "在本例中的值": "主要读取 deployment.bus_id",
            "作用": "根据 agent bus 把负荷或发电注入写入 pandapower 网络并跑潮流。",
        },
        {
            "模块": "GridEnv / safety_projector",
            "输入或输出": "消费者",
            "在本例中的值": "复用同一套容量、功率和 bus 配置",
            "作用": "环境和安全投影器用同一个部署合同，避免训练环境和安全约束不一致。",
        },
    ]
)

display(Markdown("**阅读顺序：先看 `resolve_fixed_battery_spec` 的数学计算，再看 `build_agent_deployments` 如何把 cfg 装配成列表，最后看 GridCore 如何消费这些 deployment。**"))
display(relationship)

**阅读顺序：先看 `resolve_fixed_battery_spec` 的数学计算，再看 `build_agent_deployments` 如何把 cfg 装配成列表，最后看 GridCore 如何消费这些 deployment。**

,模块,输入或输出,在本例中的值,作用
0,cfg.env / cfg.grid,输入,"num_agents=3, bus=[12,4,2], capacity=[10,12,8], C-rate=0.5",提供配置原料。
1,envs.grid.deployments,转换层,3 个 AgentDeployment,把配置整理成每个 agent 的电网接入点和电池合同。
2,GridCore,消费者,主要读取 deployment.bus_id,根据 agent bus 把负荷或发电注入写入 pandapower 网络并跑潮流。
3,GridEnv / safety_projector,消费者,复用同一套容量、功率和 bus 配置,环境和安全投影器用同一个部署合同，避免训练环境和安全约束不一致。
